# 03 — Feature engineering

Build leakage-safe predictors and 24-hour target vectors for each station.

**Inputs:** `data/processed/{station_id}_train.parquet`, `{station_id}_test.parquet`  
**Outputs:** separate train-derived and test-derived feature artifacts in `data/processed/`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import STATION_IDS
from src.feature_engineering import (
    DEFAULT_FEATURE_CONFIG,
    build_feature_frame,
    feature_column_names,
    target_column_names,
    write_feature_artifacts,
)

PROCESSED_DIR = Path("data/processed")

## Leakage contract

Each physical artifact is transformed independently. Train and sealed-test rows are never concatenated or passed together for feature calculation, so the beginning of the test artifact cannot inherit train lookbacks. Every predictor uses information available through issue time $t$: positive shifts create lags, and trailing windows include $t$, require a complete window, and propagate source missingness. Future weather is never used.

Targets are `water_level` at $t+1$ through $t+24$. `target_valid` is calculated independently inside each physical artifact and requires the full future target window to be observed and non-imputed; otherwise all 24 target values are null. No rows are dropped, no missing values are filled, no scaling is applied, and no `features_valid` shortcut is introduced. The sealed-test feature file is generated for later final evaluation but remains unavailable to stages 04–06.

In [ ]:
summaries: list[dict[str, object]] = []
failures: dict[str, str] = {}

for station_id in STATION_IDS:
    train_source_path = PROCESSED_DIR / f"{station_id}_train.parquet"
    test_source_path = PROCESSED_DIR / f"{station_id}_test.parquet"
    try:
        # Build each side in its own call; neither calculation can see the other frame.
        train_source = pd.read_parquet(train_source_path)
        train_features = build_feature_frame(
            train_source, station_id=station_id, config=DEFAULT_FEATURE_CONFIG
        )
        test_source = pd.read_parquet(test_source_path)
        test_features = build_feature_frame(
            test_source, station_id=station_id, config=DEFAULT_FEATURE_CONFIG
        )
        manifest = write_feature_artifacts(
            train_features,
            test_features,
            station_id=station_id,
            train_source_path=train_source_path,
            test_source_path=test_source_path,
            output_dir=PROCESSED_DIR,
            config=DEFAULT_FEATURE_CONFIG,
        )
        for artifact_name, frame in (
            ("train_features", train_features),
            ("test_features", test_features),
        ):
            profile = manifest["artifacts"][artifact_name]
            summaries.append(
                {
                    "station_id": station_id,
                    "artifact": artifact_name,
                    "rows": len(frame),
                    "columns": len(frame.columns),
                    "target_valid_rows": int(frame["target_valid"].sum()),
                    "start_utc": profile["timestamp_range"]["start_utc"],
                    "end_utc": profile["timestamp_range"]["end_utc"],
                    "path": profile["path"],
                }
            )
    except Exception as error:  # noqa: BLE001 -- aggregate every station failure
        failures[station_id] = f"{type(error).__name__}: {error}"

if failures:
    details = "\n".join(
        f"- {station_id}: {message}" for station_id, message in failures.items()
    )
    raise RuntimeError(
        f"Feature generation failed for {len(failures)} station(s):\n{details}"
    )

In [ ]:
artifact_summary = pd.DataFrame(summaries)
contract_summary = pd.DataFrame(
    {
        "contract": ["predictors", "targets", "forecast horizon (hours)"],
        "count": [
            len(feature_column_names(DEFAULT_FEATURE_CONFIG)),
            len(target_column_names(DEFAULT_FEATURE_CONFIG)),
            DEFAULT_FEATURE_CONFIG.horizon_hours,
        ],
    }
)
display(artifact_summary)
display(contract_summary)

Feature artifacts and standalone lineage manifests were regenerated successfully for every configured station.